In [ ]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [ ]:
print(f'Latest run date: {dt.datetime.today()}')

### Functions

In [ ]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [ ]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

### Output directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [ ]:
str_filepath = './sql/query_normal.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

### Write into df

In [ ]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# make col
df['bktype'] = 'nobk'

# show
df

### Read query

In [ ]:
str_filepath = './sql/query_bk.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

### Write into df

In [ ]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df_tmp = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# make col
df_tmp['bktype'] = 'bk'

# show
df_tmp

### Concatenate

In [ ]:
df = pd.concat([df, df_tmp])
del df_tmp
# show
df

### Convert funding month and month end date to first of each month

In [ ]:
df['dtmFunded_first'] = df['dtmFunded'].dt.to_period('M').dt.to_timestamp()
df['MonthEndDate'] = pd.to_datetime(df['MonthEndDate'])
df['MonthEndDate_first'] = df['MonthEndDate'].dt.to_period('M').dt.to_timestamp()
df

### Get months on books

In [ ]:
df['years'] = df['MonthEndDate_first'].dt.year - df['dtmFunded_first'].dt.year
# convert to months
df['months'] = df['years'] * 12
# get difference in months
df['months_tmp'] = df['MonthEndDate_first'].dt.month - df['dtmFunded_first'].dt.month
# get mob
df['months_on_books'] = df['months'] + df['months_tmp']
# show
df

### Save as parquet

In [ ]:
%%time

# save
str_filename = 'df_loss.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

### Upload to s3

In [ ]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

### Clean-up

In [ ]:
os.remove(str_local_path)